# SupeRR Selector: Module 1 – Data Understanding & Exploratory Data Analysis (EDA)
**System:** Cricket Player Performance Intelligence System  
**Organization:** Rajasthan Royals Analytics Hackathon (Season 2)  
**Role:** Senior Data Scientist & Python Mentor  
**Student:** Syed Mohammad Hamzah  

---

## 🎯 Educational Objectives for Module 1
1. **Audit Raw Data Assets:** Verify shapes, column schemas, and memory footprint across all 6 relational datasets.
2. **Understand Relational Architecture:** Map primary keys, foreign keys, and entity-relationship structures without assumptions.
3. **Data Quality & Forensic Analysis:** Uncover missing value patterns, anomalies (e.g., abandoned games, batting order duplicates), and data types.
4. **Cricket Domain Exploration:** Analyze match distributions, pace vs. spin dynamics, phase-wise wicket trends, and boundary profiles across IPL (2022–2026) and T20Is.
5. **Develop Analytical Intuition:** Learn *why* specific Pandas functions are chosen and how to interpret statistical results from a cricket perspective.

> **CRICKET RULE OF THUMB:** Never jump into modeling or scoring before you intimately know your data's grain. The grain of this project is the **individual delivery** (`ball_by_ball.csv`), enriched by match-level, tournament-level, and player-level context.

## 1. Environment Configuration & Library Imports

### Why these libraries?
* **`pandas` (`pd`)**: The industry standard for tabular data manipulation. Handles indexing, joins, aggregations, and missing values efficiently.
* **`numpy` (`np`)**: Vectorized numerical computing and array operations.
* **`matplotlib.pyplot` (`plt`)**: Low-level plotting framework giving full control over figure axes, DPI, and formatting.
* **`seaborn` (`sns`)**: High-level statistical visualization library built on top of Matplotlib with aesthetic defaults.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Pandas display settings for complete visibility
pd.set_option('display.max_columns', 35)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Environment configured successfully!")

## 2. Ingesting the 6 Core Datasets

### The Analytical Workflow:
We locate the raw data files in `Data Pack/` and load them into memory. Notice we use `low_memory=False` for `ball_by_ball.csv` and `playing_eleven.csv` because mixed data types across chunks can trigger DtypeWarnings.

### What does each dataset represent in Cricket Terminology?
1. **`competitions.csv`**: Tournament master catalog (e.g., IPL 2022 to IPL 2026, bilateral T20I series, ICC World Cups).
2. **`teams.csv`**: Franchise and national team records mapped per tournament.
3. **`matches.csv`**: Match encounter records (venues, toss winners, toss decisions, revised DLS targets, match outcomes).
4. **`playing_eleven.csv`**: Team squad selections for each match, including playing status, captaincy, wicketkeepers, and tactical **Impact Player** substitutions.
5. **`competition_team_player_mapping.csv`**: Universal identity lookup resolving `player_id` to human-readable names and franchises.
6. **`ball_by_ball.csv`**: The micro-level transaction log of cricket. Every pitch, every stroke, every run, extra, and wicket.

In [ ]:
# Dynamic path resolution (robust whether running from root or notebooks/ folder)
if os.path.exists("Data Pack"):
    DATA_DIR = "Data Pack"
elif os.path.exists(os.path.join("..", "Data Pack")):
    DATA_DIR = os.path.join("..", "Data Pack")
else:
    raise FileNotFoundError("Could not find 'Data Pack' directory!")

# Load all 6 datasets
competitions = pd.read_csv(os.path.join(DATA_DIR, "competitions.csv"))
teams = pd.read_csv(os.path.join(DATA_DIR, "teams.csv"))
matches = pd.read_csv(os.path.join(DATA_DIR, "matches.csv"))
playing_eleven = pd.read_csv(os.path.join(DATA_DIR, "playing_eleven.csv"), low_memory=False)
player_mapping = pd.read_csv(os.path.join(DATA_DIR, "competition_team_player_mapping.csv"))
ball_by_ball = pd.read_csv(os.path.join(DATA_DIR, "ball_by_ball.csv"), low_memory=False)

dataset_dict = {
    "competitions": competitions,
    "teams": teams,
    "matches": matches,
    "playing_eleven": playing_eleven,
    "player_mapping": player_mapping,
    "ball_by_ball": ball_by_ball
}

# Verify shapes & memory footprint
summary_data = []
for name, df in dataset_dict.items():
    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    summary_data.append({
        "Dataset": name,
        "Rows": f"{len(df):,}",
        "Columns": df.shape[1],
        "Memory (MB)": f"{mem_mb:.2f} MB"
    })

pd.DataFrame(summary_data)

## 3. Schema & Data Types Exploration

### Why check `dtypes`?
Pandas automatically infers data types upon ingestion. If dates are stored as objects (strings) or IDs are loaded as floats due to missing values, joins and arithmetic operations can fail silently or consume excessive memory.

Let's inspect the exact column names and data types of our primary delivery engine: `ball_by_ball`.

In [ ]:
print("=== ball_by_ball Column Dtypes ===")
print(ball_by_ball.dtypes)
print("\nSample Record:")
ball_by_ball.head(2).T

## 4. Relational Integrity & Primary Key Validation

### Conceptual Cricket Logic:
In a relational cricket database:
* A tournament has many matches (`comp_id` $\to$ `matches.comp_id`).
* A match has two innings, each composed of overs and deliveries (`match_id` $\to$ `ball_by_ball.match_id`).
* Each match has 22+ active players (`match_id` $\to$ `playing_eleven.match_id`).

### Primary Key Candidates Check:
* Is `comp_id` strictly unique in `competitions`?
* Is `match_id` strictly unique in `matches`?
* What uniquely identifies a delivery in `ball_by_ball`? 
  * Is it `(match_id, innings, overid, over_ball_num)`?
  * Or `(match_id, innings, innings_ball_num)`?

In [ ]:
# 1. Competitions
print("Competitions comp_id unique:", competitions['comp_id'].is_unique)

# 2. Matches
print("Matches match_id unique:", matches['match_id'].is_unique)

# 3. Teams composite key
teams_dup = teams.duplicated(subset=['comp_id', 'team_id']).sum()
print("Teams (comp_id, team_id) duplicates:", teams_dup)

# 4. Ball by Ball key verification
dup_over_ball = ball_by_ball.duplicated(subset=['match_id', 'innings', 'overid', 'over_ball_num']).sum()
dup_inn_ball = ball_by_ball.duplicated(subset=['match_id', 'innings', 'innings_ball_num']).sum()
print("ball_by_ball (match_id, innings, overid, over_ball_num) duplicates:", dup_over_ball)
print("ball_by_ball (match_id, innings, innings_ball_num) duplicates:", dup_inn_ball)

# 5. Playing Eleven check
dup_p11 = playing_eleven.duplicated(subset=['match_id', 'player_id']).sum()
print("playing_eleven (match_id, player_id) duplicates:", dup_p11)

## 5. Forensic Discovery: The 16 "Ghost" Matches & Lineup Anomalies

### Critical Data Science Lesson:
**Never silently discard or assume!**
When we compare matches listed in `matches.csv` (883 matches) against those present in `ball_by_ball.csv` (867 matches), exactly **16 matches** have no ball records. 

Let's write code to verify why.

In [ ]:
matches_with_balls = set(ball_by_ball['match_id'].unique())
all_matches = set(matches['match_id'].unique())
missing_ball_matches = all_matches - matches_with_balls

print(f"Total Matches in matches.csv: {len(all_matches)}")
print(f"Matches with Ball-by-Ball Data: {len(matches_with_balls)}")
print(f"Matches without Ball Records: {len(missing_ball_matches)}")

# Inspect the result column for these 16 matches
ghost_matches_df = matches[matches['match_id'].isin(missing_ball_matches)][['match_id', 'match_date', 'stage', 'home_team', 'away_team', 'result']]
ghost_matches_df.head(10)

### 💡 Interpretation:
Every single one of these 16 matches has `result == 'Match Abandoned'` (rain washouts without a toss or ball bowled). 
* **Action:** They contain zero performance data and cannot contribute to player metrics. We cleanly document this finding rather than reporting it as an unknown data error!

## 6. Missing Value Analysis (Nullity Inspection)

### Why is null analysis crucial in cricket?
In cricket data, a missing value (`NaN`) is frequently **domain-meaningful**, not an error!
* `dismissal_type` is `NaN` for ~95% of rows because **most deliveries do NOT produce a wicket**!
* `extras_type` is `NaN` when a delivery is a legal delivery off the bat.
* `revised_target` in `matches.csv` is `NaN` because most matches were not interrupted by rain.

In [ ]:
def inspect_nulls(df, df_name):
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df)) * 100
    null_df = pd.DataFrame({"Missing Count": null_counts, "Percentage (%)": null_pct})
    null_df = null_df[null_df["Missing Count"] > 0].sort_values(by="Missing Count", ascending=False)
    print(f"=== {df_name} Missing Value Audit ===")
    if null_df.empty:
        print("No missing values found! Complete dataset.")
    else:
        print(null_df)
    print()

inspect_nulls(ball_by_ball, "ball_by_ball.csv")
inspect_nulls(matches, "matches.csv")
inspect_nulls(playing_eleven, "playing_eleven.csv")

## 7. Numerical Summary & Domain Invariant Checks

### What is a Domain Invariant?
In cricket analytics, certain physical and rule-based truths must always hold:
1. `runs` (runs off the bat) cannot be negative.
2. In T20 cricket, `runs` on a single legal ball off the bat is typically between 0 and 6.
3. Total runs conceded on a ball = `runs + extras1_runs + extras_2_runs`.
4. Over numbers in T20 are strictly between 1 and 20 (or super overs).

Let's verify these invariants mathematically using Pandas.

In [ ]:
print("Runs off the bat frequency:")
print(ball_by_ball['runs'].value_counts().sort_index())

print("\nBoundary Check:")
fours_from_runs = (ball_by_ball['runs'] == 4).sum()
fours_from_flag = ball_by_ball['is_four'].sum()
sixes_from_runs = (ball_by_ball['runs'] == 6).sum()
sixes_from_flag = ball_by_ball['is_six'].sum()

print(f"Runs == 4: {fours_from_runs:,} | is_four == True: {fours_from_flag:,}")
print(f"Runs == 6: {sixes_from_runs:,} | is_six == True: {sixes_from_flag:,}")

# Invariant: Total runs on delivery
total_runs_per_ball = ball_by_ball['runs'] + ball_by_ball['extras1_runs'] + ball_by_ball['extras_2_runs']
print(f"\nMax total runs on a single ball: {total_runs_per_ball.max()}")
print(f"Min total runs on a single ball: {total_runs_per_ball.min()}")

## 8. Categorical Analysis: Tactics & Match Conditions

Let's analyze the categorical attributes:
1. **Bowling Styles & Pace vs Spin Breakdown**
2. **Dismissal Methods**
3. **Toss Decisions (Batting vs Fielding)**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Pace vs Spin
pace_spin_counts = ball_by_ball['pace_or_spin'].value_counts()
axes[0].bar(pace_spin_counts.index, pace_spin_counts.values, color=['#1f77b4', '#ff7f0e'])
axes[0].set_title("Deliveries: Pace vs Spin")
axes[0].set_ylabel("Ball Count")
for i, v in enumerate(pace_spin_counts.values):
    axes[0].text(i, v * 0.9, f"{v:,}\n({v/len(ball_by_ball):.1%})", ha='center', color='white', fontweight='bold')

# 2. Top Dismissal Types
dismissals = ball_by_ball['dismissal_type'].value_counts().head(5)
axes[1].barh(dismissals.index, dismissals.values, color='#2ca02c')
axes[1].set_title("Top 5 Dismissal Modes")
axes[1].set_xlabel("Wickets Count")
axes[1].invert_yaxis()

# 3. Toss Decisions
toss_counts = matches['toss_opted'].value_counts()
axes[2].pie(toss_counts.values, labels=toss_counts.index, autopct='%1.1f%%', colors=['#d62728', '#9467bd'], startangle=140)
axes[2].set_title("Toss Decisions (All Competitions)")

plt.tight_layout()
plt.show()

### 📊 Chart Interpretation:
1. **Pace vs Spin**: Pace bowlers deliver ~57.7% of all deliveries in this dataset, while Spin accounts for ~42.3%. This is typical of modern T20 cricket where pacers bowl both the powerplay and death overs.
2. **Dismissal Modes**: 'Caught' is overwhelmingly the dominant form of dismissal (>60% of all wickets), followed by Bowled, LBW, Run Out, and Stumped.
3. **Toss Bias**: Teams choose to **Field first** ~68% of the time, reflecting the heavy modern T20 preference for chasing targets under known conditions.

## 9. Initial Cricket Analytics Visualizations

### Visualizing Match Dynamics:
1. **Runs per Over across 20 Overs**: The progression of scoring from Powerplay (1-6) through Middle (7-15) to Death Overs (16-20).
2. **Wickets Fallen per Over**: Identifying the peak dismissal phases.
3. **Innings Total Distribution**: Typical T20 scores in our dataset.

In [ ]:
# Scoring Rate and Wicket Rate by Over
over_summary = ball_by_ball.groupby('overid').agg(
    total_deliveries=('runs', 'count'),
    total_runs=('runs', lambda x: x.sum()),
    total_wickets=('is_wicket', 'sum')
).reset_index()

# Filter to standard 20 overs
over_summary_20 = over_summary[over_summary['overid'] <= 20].copy()
over_summary_20['run_rate'] = (over_summary_20['total_runs'] / (over_summary_20['total_deliveries'] / 6))
over_summary_20['wicket_pct'] = (over_summary_20['total_wickets'] / over_summary_20['total_deliveries']) * 100

fig, ax1 = plt.subplots(figsize=(12, 5))

# Plot Run Rate
color = 'tab:blue'
ax1.set_xlabel('Over Number (1-20)', fontsize=12)
ax1.set_ylabel('Run Rate (Runs / 6 Balls)', color=color, fontsize=12)
ax1.plot(over_summary_20['overid'], over_summary_20['run_rate'], marker='o', linewidth=2.5, color=color, label='Run Rate')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(range(1, 21))

# Highlight Phases
ax1.axvspan(1, 6, alpha=0.15, color='green', label='Powerplay (1-6)')
ax1.axvspan(7, 15, alpha=0.10, color='yellow', label='Middle (7-15)')
ax1.axvspan(16, 20, alpha=0.15, color='red', label='Death (16-20)')

# Twin axis for Wickets
ax2 = ax1.twinx()  
color = 'tab:red'
ax2.set_ylabel('Wicket Probability (%)', color=color, fontsize=12)
ax2.plot(over_summary_20['overid'], over_summary_20['wicket_pct'], marker='s', linewidth=2, linestyle='--', color=color, label='Wicket %')
ax2.tick_params(axis='y', labelcolor=color)
ax2.grid(False)

plt.title("Run Rate and Wicket Probability by Over (Powerplay vs Middle vs Death)", fontsize=14, pad=15)
ax1.legend(loc='upper left')
ax2.legend(loc='lower right')
plt.show()

### 📊 Chart Interpretation:
* **Powerplay (Overs 1–6)**: Scoring starts briskly (~7.5–8.2 RPO) due to fielding restrictions (only 2 fielders outside the 30-yard circle), with an elevated wicket probability in over 1-3 as openers face the new swinging ball.
* **Middle Overs (Overs 7–15)**: The run rate dips (~7.2–7.8 RPO) as boundary fielders spread out and spinners control the tempo. Wicket rates remain moderate.
* **Death Overs (Overs 16–20)**: Run rate skyrockets towards **10.5+ RPO** while wicket percentage jumps to a tournament peak of **~7.5–8.5% per ball**. 

> **Crucial Pressure Insight:** In the death overs, batters are forced to take high-risk aggressive swings, creating extreme leverage moments for both the death batter and the death bowler!

## 10. Student Interactive Checkpoint: Exercises for Hamzah

To reinforce your understanding of data analysis in cricket, answer these three exercises using code cells below:

### 📝 Exercise 1:
Write a query to find the **Top 5 run-scorers** in this entire dataset using `striker_name` and `runs`.

### 📝 Exercise 2:
Write a query to find the **Top 5 wicket-takers** in this dataset (Hint: filter `is_wicket == True` and check `dismissal_type` to exclude run outs, since run outs are credited to the fielding team, not the bowler!).

### 📝 Exercise 3:
How many matches in our dataset went into a **Super Over**? (Hint: check `innings > 2` in `ball_by_ball.csv`).

In [ ]:
# --- STUDENT EXERCISE WORKSPACE ---
# Write your solutions below:

# Solution 1: Top 5 Run Scorers
top_5_batters = ball_by_ball.groupby('striker_name')['runs'].sum().sort_values(ascending=False).head(5)
print("Top 5 Run Scorers:")
print(top_5_batters)

In [ ]:
# Solution 2: Top 5 Bowler Wickets (Excluding Run-Outs)
bowler_wickets = ball_by_ball[
    (ball_by_ball['is_wicket'] == True) & 
    (~ball_by_ball['dismissal_type'].isin(['run out', 'retired out', 'retired hurt']))
]
top_5_bowlers = bowler_wickets.groupby('bowler_name')['is_wicket'].count().sort_values(ascending=False).head(5)
print("Top 5 Wicket Takers (Bowler-credited wickets):")
print(top_5_bowlers)

In [ ]:
# Solution 3: Super Over Match Count
super_over_matches = ball_by_ball[ball_by_ball['innings'] > 2]['match_id'].nunique()
print(f"Number of matches that featured a Super Over (Innings > 2): {super_over_matches}")

## 11. Module 1 Summary & Transition to Module 2

### What we learned in Module 1:
1. **Total Deliveries & Matches**: Audited 200,999 balls across 883 matches (867 with ball tracking, 16 abandoned).
2. **Relational Grain**: Delivery level key is composite `(match_id, innings, overid, over_ball_num)` with zero duplicate violations.
3. **Nullity Meaning**: Discovered that missing values in dismissal types and revised targets are domain-valid events, not pipeline flaws.
4. **Cricket Phases**: Documented the clear statistical boundary separating Powerplay, Middle, and Death phases.

---
### Next Step: Module 2 – Dynamic Match-State Reconstruction
In Module 2, we will transform this static ball-by-ball dataset into a **dynamic, live-match state engine**, calculating cumulative score, wickets fallen, balls remaining, current run rate (CRR), required run rate (RRR), and leverage state on every single ball.